# Tupel Modulo 12 Generator

Dieses Notebook erzeugt eine CSV-Datei mit allen Zahlen von 1 bis N,
die genau 4 verschiedene Primfaktoren haben, wobei jeder Primfaktor
aus einer der Restklassen {1, 5, 7, 11} modulo 12 stammt und alle
vier Restklassen vertreten sind.

**CSV-Format:**
- Erste Spalte (`n`): Die natürliche Zahl
- Weitere Spalten (`r1`, `r2`, `r3`, ...): Die Restklassen der Primfaktoren (1, 5, 7, 11) in der Reihenfolge ihres Erscheinens

**Beispiel:** Für `n = 5 * 13 * 7 * 11 = 5005`:
- `n = 5005`
- `r1 = 5` (5 mod 12 = 5)
- `r2 = 1` (13 mod 12 = 1)
- `r3 = 7` (7 mod 12 = 7)
- `r4 = 11` (11 mod 12 = 11)

---

## 📋 Anleitung zur Ausführung

**Wichtig:** Dieses Notebook benötigt den **SageMath-Kernel**!

### Schritte:
1. **Stellen Sie sicher, dass der SageMath-Kernel aktiv ist** (oben rechts im Notebook)
2. **Führen Sie die Zellen der Reihe nach aus** (Shift+Enter für jede Zelle)
3. **Beginnen Sie mit Zelle 1** (SageMath-Imports)
4. **Führen Sie die restlichen Zellen nacheinander aus**
5. **In der letzten Zelle können Sie N eingeben und die Berechnung starten**

---

In [21]:
# SageMath Imports
from sage.all import *
import csv
import sys

print("✓ SageMath erfolgreich geladen!")
print(f"SageMath Version: {version()}")
print(f"Python Version: {sys.version}")
sys.stdout.flush()

✓ SageMath erfolgreich geladen!
SageMath Version: SageMath version 10.8, Release Date: 2025-12-18
Python Version: 3.13.7 (main, Dec 26 2025, 08:28:22) [Clang 17.0.0 (clang-1700.3.19.1)]


## Konfiguration

In [22]:
# Erlaubte Restklassen modulo 12
ALLOWED_RESIDUES = [1, 5, 7, 11]

# Obergrenze N (kann hier geändert werden oder in der letzten Zelle interaktiv)
n = 1000

# Ausgabedatei
output_file = "quadratfreie_tupel_mod12.csv"

print(f"Konfiguration:")
print(f"  Obergrenze N: {n}")
print(f"  Erlaubte Restklassen: {ALLOWED_RESIDUES}")
print(f"  Ausgabedatei: {output_file}")
sys.stdout.flush()

Konfiguration:
  Obergrenze N: 1000
  Erlaubte Restklassen: [1, 5, 7, 11]
  Ausgabedatei: quadratfreie_tupel_mod12.csv


## Funktionen

In [23]:
def is_allowed_mod12(n):
    """
    Prüft, ob alle Primfaktoren von n in den erlaubten Restklassen {1, 5, 7, 11} mod 12 liegen.
    """
    if n == 1:
        return True
    
    fac = factor(n)
    for p, e in fac:
        if p % 12 not in ALLOWED_RESIDUES:
            return False
    return True

def get_factorization_string(n):
    """
    Erzeugt eine String-Darstellung der Faktorisierung.
    Beispiel: 5^3*17*29^2
    """
    fac = factor(n)
    parts = []
    for p, e in sorted(fac):
        if e == 1:
            parts.append(str(p))
        else:
            parts.append(f"{p}^{e}")
    return "*".join(parts)

def get_residue_sequence(n):
    """
    Gibt die Restklassen der Primfaktoren in der Reihenfolge ihres Erscheinens zurück.
    Beispiel: n = 5*13*7*11 -> [5, 1, 7, 11] (da 5 mod 12 = 5, 13 mod 12 = 1, 7 mod 12 = 7, 11 mod 12 = 11)
    """
    if n == 1:
        return []
    
    fac = factor(n)
    residues = []
    for p, e in sorted(fac):
        # Wiederhole die Restklasse entsprechend der Vielfachheit
        residue = p % 12
        residues.extend([residue] * e)
    return residues

print("✓ Funktionen definiert")
sys.stdout.flush()

✓ Funktionen definiert


## Hauptausführung

Führen Sie diese Zelle aus, um die Berechnung zu starten. Sie können `n` und `output_file` in der Konfigurationszelle ändern oder hier direkt anpassen.

In [24]:
# Hauptausführung
print(f"=== Quadratfreie Tupel Modulo 12 Generator ===")
print(f"Obergrenze: n = {n}")
print(f"SageMath Version: {version()}\n")
sys.stdout.flush()

results = []
count = 0

print("Durchsuche Zahlen von 1 bis {}...".format(n))
sys.stdout.flush()

for num in range(1, n + 1):
    # Prüfe auf Quadratfreiheit
    if not is_squarefree(num):
        continue
    
    # Prüfe, ob alle Primfaktoren in erlaubten Restklassen sind
    if not is_allowed_mod12(num):
        continue
    
    # Zahl gefunden - füge zur Liste hinzu
    residues = get_residue_sequence(num)
    
    # Erstelle Dictionary mit natürlicher Zahl und Restklassen
    record = {"n": num}
    # Füge Restklassen in der Reihenfolge ihres Erscheinens hinzu
    for i, r in enumerate(residues, 1):
        record[f"r{i}"] = r
    
    results.append(record)
    count += 1
    
    # Fortschrittsanzeige alle 10000 Zahlen
    if num % 10000 == 0:
        print(f"  Fortschritt: {num}/{n} geprüft, {count} gefunden...")
        sys.stdout.flush()

print(f"\nGefunden: {count} quadratfreie Zahlen mit Primfaktoren in {{1, 5, 7, 11}} mod 12")
sys.stdout.flush()

# Bestimme maximale Anzahl von Restklassen-Spalten
max_residues = 0
for rec in results:
    # Zähle Restklassen-Spalten (alle Spalten außer "n")
    residue_count = len([k for k in rec.keys() if k.startswith("r")])
    max_residues = max(max_residues, residue_count)

# Erstelle Spaltennamen: n, r1, r2, r3, ...
fieldnames = ["n"] + [f"r{i}" for i in range(1, max_residues + 1)]

# Schreibe CSV-Datei
print(f"\nSchreibe Ergebnisse nach '{output_file}'...")
sys.stdout.flush()
with open(output_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)

print(f"✓ Fertig! {count} Einträge in '{output_file}' geschrieben.")
print(f"CSV-Format: Erste Spalte 'n' (natürliche Zahl), dann Restklassen r1, r2, ...")
sys.stdout.flush()

# Zeige erste 10 Einträge als Beispiel
if results:
    print("\nErste 10 Einträge:")
    print("-" * 80)
    for i, rec in enumerate(results[:10], 1):
        n_val = rec["n"]
        residues_str = ", ".join([str(rec.get(f"r{j}", "")) for j in range(1, max_residues + 1) if f"r{j}" in rec])
        fac_str = get_factorization_string(n_val)
        print(f"{i:3d}. n={n_val:6d} | Restklassen: [{residues_str}] | Faktorisierung: {fac_str}")
    if len(results) > 10:
        print(f"... und {len(results) - 10} weitere Einträge")
sys.stdout.flush()

=== Quadratfreie Tupel Modulo 12 Generator ===
Obergrenze: n = 1000
SageMath Version: SageMath version 10.8, Release Date: 2025-12-18

Durchsuche Zahlen von 1 bis 1000...

Gefunden: 303 quadratfreie Zahlen mit Primfaktoren in {1, 5, 7, 11} mod 12

Schreibe Ergebnisse nach 'quadratfreie_tupel_mod12.csv'...
✓ Fertig! 303 Einträge in 'quadratfreie_tupel_mod12.csv' geschrieben.
CSV-Format: Erste Spalte 'n' (natürliche Zahl), dann Restklassen r1, r2, ...

Erste 10 Einträge:
--------------------------------------------------------------------------------
  1. n=     1 | Restklassen: [] | Faktorisierung: 
  2. n=     5 | Restklassen: [5] | Faktorisierung: 5
  3. n=     7 | Restklassen: [7] | Faktorisierung: 7
  4. n=    11 | Restklassen: [11] | Faktorisierung: 11
  5. n=    13 | Restklassen: [1] | Faktorisierung: 13
  6. n=    17 | Restklassen: [5] | Faktorisierung: 17
  7. n=    19 | Restklassen: [7] | Faktorisierung: 19
  8. n=    23 | Restklassen: [11] | Faktorisierung: 23
  9. n=    29 | 

## Interaktive Version (optional)

Falls Sie interaktive Widgets verwenden möchten, können Sie diese Zelle ausführen:

In [25]:
# Optional: Interaktive Widgets
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    
    n_input = widgets.BoundedIntText(
        value=1000,
        min=1,
        max=10**8,
        step=1000,
        description='Obergrenze N:',
        style={'description_width': 'initial'}
    )
    
    filename_input = widgets.Text(
        value='quadratfreie_tupel_mod12.csv',
        description='Ausgabedatei:',
        style={'description_width': 'initial'}
    )
    
    run_button = widgets.Button(
        description='Berechnung starten',
        button_style='success',
        icon='play'
    )
    
    output_area = widgets.Output()
    
    def on_button_click(b):
        with output_area:
            clear_output(wait=True)
            n_val = n_input.value
            filename = filename_input.value
            
            # Verwende die Hauptfunktion
            print(f"Starte Berechnung für N = {n_val}...")
            print(f"Ergebnis wird in '{filename}' gespeichert.\n")
            
            # Temporär n und output_file setzen
            global n, output_file
            n_old = n
            output_file_old = output_file
            n = n_val
            output_file = filename
            
            try:
                # Führe Berechnung aus (kopiere Code aus vorheriger Zelle)
                results = []
                count = 0
                
                for num in range(1, n + 1):
                    if not is_squarefree(num):
                        continue
                    if not is_allowed_mod12(num):
                        continue
                    
                    residues = get_residue_sequence(num)
                    record = {"n": num}
                    for i, r in enumerate(residues, 1):
                        record[f"r{i}"] = r
                    results.append(record)
                    count += 1
                    
                    if num % 10000 == 0:
                        print(f"  Fortschritt: {num}/{n} geprüft, {count} gefunden...")
                
                print(f"\nGefunden: {count} quadratfreie Zahlen")
                
                max_residues = 0
                for rec in results:
                    residue_count = len([k for k in rec.keys() if k.startswith("r")])
                    max_residues = max(max_residues, residue_count)
                
                fieldnames = ["n"] + [f"r{i}" for i in range(1, max_residues + 1)]
                
                with open(output_file, "w", newline="", encoding="utf-8") as f:
                    writer = csv.DictWriter(f, fieldnames=fieldnames)
                    writer.writeheader()
                    writer.writerows(results)
                
                print(f"✓ Fertig! {count} Einträge in '{output_file}' geschrieben.")
                
                if results:
                    print("\nErste 10 Einträge:")
                    for i, rec in enumerate(results[:10], 1):
                        n_val = rec["n"]
                        residues_str = ", ".join([str(rec.get(f"r{j}", "")) for j in range(1, max_residues + 1) if f"r{j}" in rec])
                        fac_str = get_factorization_string(n_val)
                        print(f"{i:3d}. n={n_val:6d} | Restklassen: [{residues_str}] | Faktorisierung: {fac_str}")
                
                # Wiederherstellen
                n = n_old
                output_file = output_file_old
                
            except Exception as e:
                print(f"\n✗ Fehler: {e}")
                import traceback
                traceback.print_exc()
    
    run_button.on_click(on_button_click)
    
    # Widgets anzeigen
    controls = widgets.VBox([
        widgets.HBox([n_input, filename_input]),
        run_button,
        output_area
    ])
    
    display(controls)
    print("✓ Interaktive Widgets geladen. Verwenden Sie die Steuerelemente oben.")
    
except ImportError:
    print("Hinweis: ipywidgets nicht verfügbar. Verwenden Sie die vorherige Zelle für die Berechnung.")

✓ Interaktive Widgets geladen. Verwenden Sie die Steuerelemente oben.
